In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [9]:
article_text = """
Earth is the third planet from the Sun and the only astronomical object known to the harbor life. This is enabled by Earth being an ocean world, the only one in the Solar System sustaining liquid surface water. Almost all of Earth's water is contained in its global ocean, covering 70.8% of Earth's crust. The remaining 29.2% of Earth's crust is land, most of which is located in teh form of continental landmasses within Earth's within Earth's land hemisphere. Most of Earth's land is at least somewhat ice at Earth's polar deserts retain more water than Earth's groundwater, lakes, rivers and atmospheric water combined. Earth's crust consists of slowly moving tectonic plates, which interact to produce mountain ranges, volcanoes, and earthquakes. Earth has a liquid outer core that generates a magentosphere capable of deflecting most of the destructive solar winds and cosmic radiation. 

Earth's atmosphere and oceans were formed by volcanic activity and outgassing.[43] Water vapor from
these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,
and comets.[44] Sufficient water to fill the oceans may have been on Earth since it formed.[45] In this
model, atmospheric greenhouse gases kept the oceans from freezing when the newly forming Sun had
only 70% of its current luminosity.[46] By 3.5 Ga, Earth's magnetic field was established, which helped
prevent the atmosphere from being stripped away by the solar wind.[47]
"""

In [10]:
# TODO: Read image data, feed into Claude

with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

messages = []

add_user_message(messages, [
    # {
    #     "type": "document",
    #     "source": {
    #         "type": "base64",
    #         "media_type": "application/pdf",
    #         "data": file_bytes
    #     },
    #     "title": "earth.pdf",
    #     "citations": { "enabled": True}
    # },
    {
        "type": "document",
        "source": {
            "type": "text",
            "media_type": "text/plain",
            "data": article_text,
        },
        "title": "Earth Article",
        "citations": { "enabled": True}
    },
    {
        "type": "text",
        "text": "How were the Earth's atmosphere and oceans were formed?"
    }
])

chat(messages)

Message(id='msg_01Smtn62B7rWRgJaS1sFRbSP', container=None, content=[TextBlock(citations=[CitationCharLocation(cited_text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", document_index=0, document_title='Earth Article', end_char_index=973, file_id=None, start_char_index=895, type='char_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text='[43] Water vapor from\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\nand comets.', document_index=0, document_title='Earth Article', end_char_index=1104, file_id=None, start_char_index=973, type='char_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', role='ass